# Automated Interpretability

Heuristic methods to automatically characterize what attention heads
and neurons do, without requiring human inspection of every component.

This notebook covers the `irtk.auto_interp` module.

## Why This Matters

Automated interpretability tools generate labels and summaries for heads and neurons based on their activation patterns and output effects. While not a replacement for manual analysis, they help prioritize which components to investigate and provide a starting point for understanding large models.

**Key references:**
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import auto_interp

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Auto-Label a Single Head

Classify an attention head by its behavior pattern.

In [ ]:
prompts = [
    "The Eiffel Tower is located in",
    "The capital of France is",
    "London is the capital of",
    "Berlin is a city in",
    "The president of the United States is",
]
token_seqs = [model.to_tokens(p) for p in prompts]

# Analyze L0H0
result = auto_interp.auto_label_head(model, 0, 0, token_seqs)

print(f"Head L0H0:")
print(f"  Label: {result['label']}")
print(f"  Confidence: {result['confidence']:.3f}")
print(f"  Mean entropy: {result['mean_entropy']:.3f}")
print(f"  All scores:")
for name, score in sorted(result['scores'].items(), key=lambda x: -x[1]):
    print(f"    {name}: {score:.3f}")

## 2. Head Type Classifier

Classify all heads in the model at once.

In [ ]:
result = auto_interp.head_type_classifier(model, token_seqs)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(result['confidence_matrix'], cmap='YlOrRd', aspect='auto')
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title('Head Classification Confidence')
plt.colorbar(im, ax=ax, label='Confidence')

# Annotate with labels
for l in range(model.cfg.n_layers):
    for h in range(model.cfg.n_heads):
        label = result['classifications'][f'L{l}H{h}'][:4]
        ax.text(h, l, label, ha='center', va='center', fontsize=5, color='black')

plt.tight_layout()
plt.show()

print("Type counts:")
for t, c in sorted(result['type_counts'].items(), key=lambda x: -x[1]):
    print(f"  {t}: {c}")

## 3. Auto-Label a Neuron

Characterize a neuron by its top activating contexts.

In [ ]:
result = auto_interp.auto_label_neuron(model, 6, 0, token_seqs, k=10)

print(f"Neuron L6N0:")
print(f"  Mean activation: {result['mean_activation']:.4f}")
print(f"  Max activation: {result['max_activation']:.4f}")
print(f"  Firing rate: {result['firing_rate']:.2%}")
print(f"  Top activations:")
for act, pos, pi in result['top_activations'][:5]:
    print(f"    act={act:.4f}, pos={pos}, prompt={pi}")

## 4. Feature Summary Stats

Compute activation statistics across all dimensions at a hook point.

In [ ]:
result = auto_interp.feature_summary_stats(model, "blocks.6.hook_resid_post", token_seqs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(result['mean_activations'], bins=50, color='steelblue')
axes[0,0].set_title('Mean Activation Distribution')

axes[0,1].hist(result['std_activations'], bins=50, color='coral')
axes[0,1].set_title('Std Activation Distribution')

axes[1,0].hist(result['sparsity'], bins=50, color='seagreen')
axes[1,0].set_title('Sparsity Distribution')

axes[1,1].hist(result['kurtosis'], bins=50, color='purple', range=(-10, 50))
axes[1,1].set_title('Kurtosis Distribution')

plt.tight_layout()
plt.show()

## 5. Full Component Report

Generate a summary of what each layer does.

In [ ]:
result = auto_interp.component_report(model, token_seqs)

print(f"Model: {result['n_layers']} layers, {result['n_heads']} heads\n")
for summary in result['layer_summary']:
    print(summary)

## Summary

| Function | Purpose |
|----------|--------|
| `auto_label_head()` | Classify a single attention head by behavior |
| `auto_label_neuron()` | Characterize neuron by top activating contexts |
| `feature_summary_stats()` | Statistics for all dimensions at a hook |
| `head_type_classifier()` | Classify all heads in the model |
| `component_report()` | Per-layer summary of component roles |